321 Assignment 3 - Public Ciphers

William Kim and James Irwin 

## Part 1: Implement Diffie Hellman Key Exchange

In [1]:
from Crypto.Hash import SHA256 
from Crypto.Cipher import AES
from Crypto.Util.Padding import pad, unpad
from Crypto.Random import get_random_bytes
import secrets

In [2]:
#Helper Functions. Truncating the output of SHA256 to 16 bytes, and working accordingly 
def derive_aes_key(shared_secret):
    hash_obj = SHA256.new(str(shared_secret).encode())
    return hash_obj.digest()[:16]


def aes_encrypt(key, iv, message):
    cipher = AES.new(key, AES.MODE_CBC, iv)
    ciphertext = cipher.encrypt(pad(message.encode(), 16))
    return ciphertext


def aes_decrypt(key, iv, ciphertext):
    cipher = AES.new(key, AES.MODE_CBC, iv)
    plaintext = unpad(cipher.decrypt(ciphertext), 16)
    return plaintext.decode()

We will run Diffie-Hellman on a small scale for Alice and Bob, to demonstrate how it works

In [ ]:
# Bob and Alice Keys Small setting example
q = 37
a = 5

# Alice
priv_alice = secrets.randbelow(q) #Alice private
pub_alice = pow(a,priv_alice, q) #Alice public

# Bob
priv_bob = secrets.randbelow(q) #Bob private
pub_bob = pow(a, priv_bob, q) #Bob public

print("Alice public value: ", pub_alice)
print("Bob public value: ", pub_bob)


Alice public value:  9
Bob public value:  36


We can construct a shared value for each party, using your private value, and a public value from someone else, and these shared values should be the same. 
    

In [4]:
shared_alice = pow(pub_bob, priv_alice, q)
shared_bob = pow(pub_alice, priv_bob, q)

print("Alice's secret value:", shared_alice)
print("Bob's secret value:  ", shared_bob)
print("Secrets match:", shared_alice == shared_bob)

Alice's secret value: 1
Bob's secret value:   1
Secrets match: True


Key derivation: SHA-256 to get symmetric encryption key 

In [ ]:
aes_key = derive_aes_key(shared_alice)
i_v = b'\x00' * 16  # shared initialization vector

c0 = aes_encrypt(aes_key, i_v, "Hello Bob")
print("Alice encrypts the message 'Hello Bob', to the ciphertext: ", c0)

Alice encrypts the message 'Hello Bob', to the ciphertext:  b'6\x93d\xb2uv\x06\xe7\xd8\xb6\xd6\x1a\x8e\xc2\xed\xc9'


Bob will now take Alices encrypted cipher text, and decrypt it. 

In [6]:
decrypted_aes_bob = aes_decrypt(aes_key, i_v, c0)
print ("Bob decrypts the message from the ciphertext : ", decrypted_aes_bob)

Bob decrypts the message from the ciphertext :  Hello Bob


Bob will then encrypt his own message to send back 

In [7]:
c1 = aes_encrypt(aes_key, i_v, "Hi there Alice!")

print("Bob's message 'Hi there Alice' will be encrypted to :", c1)

Bob's message 'Hi there Alice' will be encrypted to : b'\xa3\xa5\x8asjJY8\x97\xab\xfc\xd4\x0bWX\r'


Alice will now receive Bob's encrypted data, and decrypt it to receive the message

### TODO: use some "real" numbers

In [ ]:
# TODO ...

## Part 2: Implement MITM Key Fixing & Negotiated Groups

### Modify Task 1 Implementation

In [ ]:
# Alice sends q and a to Bob
q = 37
a = 5

# Alice
priv_alice = secrets.randbelow(q) #Alice private
pub_alice = pow(a,priv_alice, q) #Alice public

# Bob
priv_bob = secrets.randbelow(q) #Bob private
pub_bob = pow(a, priv_bob, q) #Bob public

print("Alice public value: ", pub_alice)
print("Bob public value: ", pub_bob)

Alice public value:  21
Bob public value:  24


In [ ]:
# at this point, Mallory intercepts and sends q to both Alice and Bob
mallory_q = 37

print("Alice public value: ", mallory_q)
print("Bob public value: ", mallory_q)

Alice public value:  37
Bob public value:  37


In [18]:
# now, the shared key is calculated using q instead of public keys, 
# resulting in a secret value of 0

shared_alice = pow(mallory_q, priv_alice, q)
shared_bob = pow(mallory_q, priv_bob, q)

print("Alice's secret value:", shared_alice)
print("Bob's secret value:  ", shared_bob)
print("Secrets match:", shared_alice == shared_bob)

Alice's secret value: 0
Bob's secret value:   0
Secrets match: True


In [26]:
# Mallory knows that math will work out this way
mallory_shared = 0

# alice generates key
aes_key = derive_aes_key(shared_alice)
i_v = b'\x00' * 16  # shared initialization vector

# mallory is able to generate the same key
mallory_aes_key = derive_aes_key(mallory_shared)

In [ ]:
# and then decrypt messages! 

# alice attempts to send message
c0 = aes_encrypt(aes_key, i_v, "Hello Bob")
print("Alice encrypts the message 'Hello Bob', to the ciphertext: ", c0)

# mallory decrypts
decrypted_aes_mallory = aes_decrypt(mallory_aes_key, i_v, c0)
print ("Mallory decrypts the message from the ciphertext : ", decrypted_aes_mallory)

Alice encrypts the message 'Hello Bob', to the ciphertext:  b'k\xcc\xf2mS\x12_3W\xf3p\xfa\x00z\xc4\xc2'
Bob decrypts the message from the ciphertext :  Hello Bob


### Repeat the Attack Using Generator a

In [31]:
# TODO: make sure that this is correct! 

# Alice sends q and a to Bob
q = 37
a = 5

# But then Mallory changes a to 1
a = 1

# Alice generates private and public keys
priv_alice = secrets.randbelow(q) #Alice private
pub_alice = pow(a,priv_alice, q) #Alice public

# Bob generate private and public keys
priv_bob = secrets.randbelow(q) #Bob private
pub_bob = pow(a, priv_bob, q) #Bob public

print("Alice public value: ", pub_alice)
print("Bob public value: ", pub_bob)

# Alice and Bob create shared keys
shared_alice = pow(pub_bob, priv_alice, q)
shared_bob = pow(pub_alice, priv_bob, q)

# Mallory knows that all private and public keys equal 1, so she is able to 
# create her own shared valye
shared_mallory = pow(1, 1, q)

print("Alice's secret value:", shared_alice)
print("Bob's secret value:  ", shared_bob)

# alice generates key
aes_key = derive_aes_key(shared_alice)
i_v = b'\x00' * 16  # shared initialization vector

# BUT MALLORY IS ABLE TO GENERATE KEY ALSO!
mallory_aes_key = derive_aes_key(shared_alice)

# alice encrypts message
c0 = aes_encrypt(aes_key, i_v, "Hello Bob")
print("Alice encrypts the message 'Hello Bob', to the ciphertext: ", c0)

# and mallory decrypts it
decrypted_aes_mallory = aes_decrypt(mallory_aes_key, i_v, c0)
print ("Mallory decrypts the message from the ciphertext : ", decrypted_aes_mallory)

Alice public value:  1
Bob public value:  1
Alice's secret value: 1
Bob's secret value:   1
Alice encrypts the message 'Hello Bob', to the ciphertext:  b'6\x93d\xb2uv\x06\xe7\xd8\xb6\xd6\x1a\x8e\xc2\xed\xc9'
Mallory decrypts the message from the ciphertext :  Hello Bob


## Part 3: Implement “Textbook” RSA & MITM Key Fixing via Malleability: 

## Questions

1. 
2. 
3. 
4.

## Citations

https://security.stackexchange.com/questions/91699/why-cant-i-mitm-a-diffie-hellman-key-exchange

https://stackoverflow.com/questions/10471009/how-does-the-man-in-the-middle-attack-work-in-diffie-hellman

